# Lecture 2 — Fully-connected networks: train on a real dataset

In the sandbox you built an **affine layer and its forward pass by hand** on toy numbers. Now you'll train the real thing: a small fully-connected network that classifies **penguin species** from four body measurements, using the **Palmer Penguins** dataset.

It is tiny and fully inspectable — only 333 penguins, four numbers each — so you can actually *see* the data, watch the loss fall, and read every mistake the model makes.

> **Runtime -> Change runtime type -> T4 GPU** before you start. (This dataset is so small that CPU is fine too, but we wire up the GPU the same way you will for every later notebook.)

## 1. Setup and GPU check

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if device == "cpu":
    print("WARNING: no GPU found. Runtime > Change runtime type > T4 GPU.")
    print("(This dataset is tiny, so CPU is totally fine here too.)")

# Make results reproducible.
torch.manual_seed(0)
np.random.seed(0)


def accuracy(logits, y):
    """Fraction correct, given raw model outputs (logits) and integer labels."""
    preds = logits.argmax(dim=1)
    return (preds == y).float().mean().item()


def plot_curves(history):
    """Plot train/val loss and validation accuracy over epochs."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(history["train_loss"], label="train loss")
    ax1.plot(history["val_loss"], label="val loss")
    ax1.set_xlabel("epoch"); ax1.set_ylabel("loss"); ax1.set_title("Loss"); ax1.legend()
    ax2.plot(history["val_acc"], color="tab:green")
    ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.set_title("Validation accuracy")
    plt.tight_layout(); plt.show()

## 2. Get the real data

**Palmer Penguins** — measurements of 344 penguins of three species (Adelie, Chinstrap, Gentoo) from islands near Antarctica. `seaborn` ships with this dataset, so there is nothing to download.

What is *real and messy* about it:

- Some penguins have **missing measurements** (a researcher did not record every field). We drop those rows — your first taste of real-data cleaning.
- The four features live on **wildly different scales**: bill length is about 45 *mm*, body mass is about 4200 *grams*. We will standardize them, and later you will see what happens if you do not.
- The classes are **imbalanced** (many Adelie, fewer Chinstrap), just like real datasets.

We predict `species` from the four numeric body measurements.

In [ ]:
df = sns.load_dataset("penguins")
print("Raw rows:", len(df))

# Real-data cleaning: drop penguins with any missing measurement.
df = df.dropna().reset_index(drop=True)
print("After dropping rows with missing values:", len(df))

feature_cols = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
target_col = "species"

# Map species names -> integer class ids (0, 1, 2).
species_names = sorted(df[target_col].unique())
species_to_idx = {name: i for i, name in enumerate(species_names)}
print("Classes:", species_to_idx)

X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_all = df[target_col].map(species_to_idx).to_numpy(dtype=np.int64)

## 3. Look at the data

Before training *anything*, look at what you are feeding the model. Seeing the data **is** the real-dataset experience.

In [ ]:
# A few raw rows.
print(df.head())

# How many of each species?
print("\nClass counts:")
print(df[target_col].value_counts())

# Scatter two features, coloured by species: can you already see the groups?
plt.figure(figsize=(6, 5))
sns.scatterplot(data=df, x="flipper_length_mm", y="bill_length_mm", hue="species")
plt.title("Penguins: flipper length vs bill length")
plt.show()

### Split first, then standardize using TRAIN statistics only

We split into **train / val / test**, then standardize each feature to mean 0, std 1.

**The subtle, important part:** we compute the mean and std on the **training set only**, and reuse those exact numbers to transform val and test. If we computed them over the whole dataset, information from val and test would leak into training and our scores would be optimistically wrong. This is **data leakage**, and it is one of the most common ways people fool themselves.

In [ ]:
from sklearn.model_selection import train_test_split

# 60% train, 20% val, 20% test. stratify keeps class proportions in each split.
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_all, y_all, test_size=0.40, random_state=0, stratify=y_all
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=0, stratify=y_tmp
)
print(f"train={len(X_train)}  val={len(X_val)}  test={len(X_test)}")

# Standardize using TRAIN statistics ONLY (no leakage from val/test).
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)
print("Train mean:", np.round(train_mean, 2))
print("Train std: ", np.round(train_std, 2))


def standardize(X):
    return (X - train_mean) / train_std


# Build tensors once. The dataset is tiny, so we use the whole thing as one batch.
Xtr = torch.tensor(standardize(X_train))
Xva = torch.tensor(standardize(X_val))
Xte = torch.tensor(standardize(X_test))
ytr = torch.tensor(y_train)
yva = torch.tensor(y_val)
yte = torch.tensor(y_test)
print("Xtr shape:", Xtr.shape, " ytr shape:", ytr.shape)

## 4. Build the model

In [ ]:
# 🔧 Your turn -- this mirrors what you built from scratch; read it (and try writing it yourself) before running.
#
# Two affine layers with a ReLU between them. This is exactly the affine layer
# and forward pass you wrote by hand in the sandbox, just stacked and run on a GPU:
#   Linear(4, 16) -> ReLU -> Linear(16, 3)
#   4 inputs (the measurements) -> 16 hidden units -> 3 outputs (one score per species)
model = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3),
).to(device)

print(model)

## 5. Train

The training loop is the same five steps every time:

1. **forward** — run inputs through the model to get logits
2. **loss** — measure how wrong they are (`CrossEntropyLoss`)
3. **zero grads** — clear last step's gradients
4. **backward** — compute new gradients (`loss.backward()`)
5. **step** — nudge the weights (`optimizer.step()`)

The model and every batch go `.to(device)`. This dataset is so small that we treat the whole training set as a single batch per epoch.

In [ ]:
# 🔧 Your turn -- this is the 5-step loop. Read each line and match it to the steps above.
def train_model(model, Xtr, ytr, Xva, yva, epochs=200, lr=0.05):
    Xtr, ytr = Xtr.to(device), ytr.to(device)
    Xva, yva = Xva.to(device), yva.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        logits = model(Xtr)            # 1. forward
        loss = loss_fn(logits, ytr)    # 2. loss
        optimizer.zero_grad()          # 3. zero grads
        loss.backward()                # 4. backward
        optimizer.step()               # 5. step

        # Track validation each epoch (no gradients needed).
        model.eval()
        with torch.no_grad():
            val_logits = model(Xva)
            val_loss = loss_fn(val_logits, yva).item()
            val_acc = accuracy(val_logits, yva)
        history["train_loss"].append(loss.item())
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if (epoch + 1) % 40 == 0:
            print(f"epoch {epoch + 1:3d} | train loss {loss.item():.3f} "
                  f"| val loss {val_loss:.3f} | val acc {val_acc:.3f}")
    return history


history = train_model(model, Xtr, ytr, Xva, yva)

## 6. Evaluate

In [ ]:
# Loss curves and validation accuracy over training.
plot_curves(history)

# Final accuracy on the held-out TEST set (numbers the model has never seen).
model.eval()
with torch.no_grad():
    test_logits = model(Xte.to(device))
test_acc = accuracy(test_logits, yte.to(device))
print(f"Test accuracy: {test_acc:.3f}")

In [ ]:
# Confusion matrix: rows = true species, columns = predicted. The diagonal is
# correct; off-diagonal cells show which species get mistaken for which.
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

test_preds = test_logits.argmax(dim=1).cpu().numpy()
cm = confusion_matrix(yte.numpy(), test_preds)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=species_names).plot(cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.show()
print(cm)

## 7. Experiment

Now change one thing at a time and watch what happens. Fill in the table at the end with **your** numbers.

### Experiment A: what if we skip standardization?

We retrain the exact same model on the **raw** features (bill length in mm sitting right next to body mass in grams). Watch the validation accuracy.

In [ ]:
torch.manual_seed(0)

# Raw features -- NO standardization.
Xtr_raw = torch.tensor(X_train)
Xva_raw = torch.tensor(X_val)

model_raw = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 3)).to(device)
hist_raw = train_model(model_raw, Xtr_raw, ytr, Xva_raw, yva)

print()
print(f"WITHOUT standardization -> final val acc: {hist_raw['val_acc'][-1]:.3f}")
print(f"WITH standardization    -> final val acc: {history['val_acc'][-1]:.3f}")

### Experiment B: add a hidden layer

Make the network **deeper**: two hidden layers instead of one. Does a bigger model help on this easy dataset?

**Beat-the-baseline target:** your one-hidden-layer model already does very well. Try to *match* it with the deeper net, and notice that "more layers" is not automatically "better", especially when the data is already easy to separate.

In [ ]:
torch.manual_seed(0)

model_deep = nn.Sequential(
    nn.Linear(4, 16), nn.ReLU(),
    nn.Linear(16, 16), nn.ReLU(),   # <- extra hidden layer
    nn.Linear(16, 3),
).to(device)
hist_deep = train_model(model_deep, Xtr, ytr, Xva, yva)

print()
print(f"1 hidden layer  -> final val acc: {history['val_acc'][-1]:.3f}")
print(f"2 hidden layers -> final val acc: {hist_deep['val_acc'][-1]:.3f}")

### Fill in your ablation table

| Setup | Final val acc | Test acc |
|---|---|---|
| Baseline (standardized, 1 hidden layer) | ? | ? |
| No standardization | ? | -- |
| 2 hidden layers | ? | -- |

*(Re-run the evaluate cell after swapping in a model to get its test accuracy.)*

## 8. Reflect: report YOUR results

1. **Test accuracy:** what test accuracy did your baseline model reach?
2. **Confusions:** look at your confusion matrix. *Which* species get mistaken for each other? Look back at the scatter plot in section 3: does that overlap explain it?
3. **Did standardizing help?** Compare the val accuracy with and without standardization. Why does putting features on the same scale matter for gradient descent?
4. **Deeper model:** did adding a hidden layer help, hurt, or do nothing? What does that tell you about matching model size to how hard the problem is?
5. **Leakage check:** we computed the standardization mean and std on the *training set only*. In one sentence, what would go wrong if we had used the whole dataset instead?